# LegalQA pipeline smoke test trên Kaggle

Notebook này kiểm tra luồng end-to-end: clone repo → cài môi trường → unit test/metric self-test → prepare → model audit → index corpus mẫu → retrieve → generate → evaluate.

Mặc định chỉ dùng **250 file corpus** và **3 câu hỏi dev** để phát hiện lỗi tích hợp; điểm số smoke không dùng để đánh giá chất lượng mô hình. Bật GPU và Internet trước khi Run All. Không có bước SFT hoặc submission.

## 1. Cấu hình smoke

Đổi `RUN_NAME` khi thay cấu hình để tránh dùng lại cache không tương thích. Đặt `USE_FULL_CORPUS=True` chỉ khi muốn kiểm tra index toàn bộ corpus.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

if not Path('/kaggle').exists():
    raise RuntimeError('Notebook smoke này chỉ được cấu hình để chạy trên Kaggle.')

REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_REF = 'main'
WORK_BASE = Path('/kaggle/working')
CODE = WORK_BASE / 'uit-dsc-2026-task2-legalqa'
RUN_NAME = 'legalqa_smoke_v1'
RUN_ROOT = WORK_BASE / RUN_NAME

USE_REPO_DATA = True
KAGGLE_DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')
USE_FULL_CORPUS = False
SMOKE_CORPUS_FILES = 250
SMOKE_QUESTIONS = 3
GENERATION_MODE = 'generate'  # đổi thành 'extractive' để kiểm tra nhanh mà không sinh bằng LLM

MODELS = RUN_ROOT / 'models'
INDEX = RUN_ROOT / 'index'
DATA = RUN_ROOT / 'data'
CFG = RUN_ROOT / 'smoke_config.json'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Smoke output:', RUN_ROOT)

## 2. Clone repo và tạo cấu hình nhẹ

Nếu repo đã được clone trong phiên Kaggle, cell chỉ cập nhật fast-forward. Dữ liệu mặc định lấy trực tiếp từ repo.

In [ ]:
if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} đã tồn tại nhưng không phải Git repo. Hãy Restart Session hoặc đổi CODE.')
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError(f'Remote không đúng repo yêu cầu: {remote}')
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CODE)], check=True)

DATASET_ROOT = CODE if USE_REPO_DATA else KAGGLE_DATASET_ROOT
TRAIN_PATH = DATASET_ROOT / 'train.json'
TEST_PATH = DATASET_ROOT / 'public-official.json'
CORPUS_PATH = DATASET_ROOT / ('selected-contexts.zip' if USE_REPO_DATA else 'selected-contexts')
for label, path in [('train', TRAIN_PATH), ('test', TEST_PATH), ('corpus', CORPUS_PATH)]:
    if not path.exists():
        raise FileNotFoundError(f'{label}: {path}')

smoke_cfg = json.loads((CODE / 'config.json').read_text(encoding='utf-8'))
smoke_cfg['retrieval'].update({
    'bm25_k': 20, 'dense_k': 20, 'pool_k': 8,
    'parents_k': 2, 'reranker_batch': 4, 'reranker_max_tokens': 512,
})
smoke_cfg['generation'].update({
    'max_input_tokens': 2048, 'max_new_tokens': 384,
    'parent_max_tokens': 640, 'min_context_tokens': 128,
})
CFG.write_text(json.dumps(smoke_cfg, ensure_ascii=False, indent=2), encoding='utf-8')

def run(*args):
    command = [sys.executable, '-m', 'legalqa', '--config', str(CFG), '--models', str(MODELS), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=CODE, check=True)

commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Code:', CODE, '| Commit:', commit)

## 3. Cài dependencies và chạy kiểm định CPU

Cell này phải hoàn tất với unit test `OK` và metric self-test không lỗi trước khi tải model.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, check=True)
subprocess.run([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, check=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_ROOT / 'environment.freeze.txt').write_text(freeze, encoding='utf-8')

## 4. Prepare dữ liệu và tạo corpus mẫu

Corpus mẫu chỉ dùng để kiểm tra pipeline có chạy xuyên suốt. Không dùng metric tạo ra từ corpus mẫu để so sánh chất lượng.

In [ ]:
run('prepare', '--train', TRAIN_PATH, '--test', TEST_PATH, '--output', DATA)

if USE_FULL_CORPUS:
    INDEX_CORPUS = CORPUS_PATH
else:
    INDEX_CORPUS = RUN_ROOT / f'corpus_{SMOKE_CORPUS_FILES}'
    marker = INDEX_CORPUS / '_smoke_corpus.json'
    if not marker.exists():
        INDEX_CORPUS.mkdir(parents=True, exist_ok=True)
        if any(INDEX_CORPUS.glob('context_*.json')):
            raise RuntimeError(f'{INDEX_CORPUS} có corpus dở dang. Hãy đổi RUN_NAME.')
        copied = []
        if CORPUS_PATH.is_file():
            with zipfile.ZipFile(CORPUS_PATH) as archive:
                members = sorted(
                    (item for item in archive.infolist() if not item.is_dir() and Path(item.filename).name.startswith('context_') and item.filename.endswith('.json')),
                    key=lambda item: item.filename,
                )[:SMOKE_CORPUS_FILES]
                for item in members:
                    target = INDEX_CORPUS / Path(item.filename).name
                    with archive.open(item) as source_file, target.open('wb') as target_file:
                        shutil.copyfileobj(source_file, target_file)
                    copied.append(target.name)
        else:
            members = sorted(CORPUS_PATH.rglob('context_*.json'))[:SMOKE_CORPUS_FILES]
            for item in members:
                target = INDEX_CORPUS / item.name
                shutil.copy2(item, target)
                copied.append(target.name)
        if not copied:
            raise RuntimeError('Không tìm thấy context_*.json trong corpus.')
        marker.write_text(json.dumps({'source': str(CORPUS_PATH), 'files': copied}, indent=2), encoding='utf-8')
    print('Corpus smoke files:', len(list(INDEX_CORPUS.glob('context_*.json'))))

print(json.loads((DATA / 'data_report.json').read_text(encoding='utf-8')))

## 5. Tải, khóa và audit model

Đây là bước cần Internet và có thể tải nhiều GB. Audit phải xác nhận tổng tham số nhỏ hơn 4 tỷ.

In [ ]:
if not (MODELS / 'models.lock.json').exists():
    run('fetch-models')
run('audit-models')
audit = json.loads((MODELS / 'parameter_audit.json').read_text(encoding='utf-8'))
assert audit['passes'] and audit['total_with_unmerged_lora'] < 4_000_000_000
print('Parameter audit OK:', f"{audit['total_with_unmerged_lora']:,}")

## 6. Build index smoke

Index hoàn chỉnh được tái sử dụng khi chạy lại cùng `RUN_NAME`.

In [ ]:
if not (INDEX / 'index_manifest.json').exists():
    run('build-index', '--corpus', INDEX_CORPUS, '--output', INDEX)
else:
    print('Reusing index:', INDEX)
print(json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8')))

## 7. Retrieve, generate và evaluate

Lấy `SMOKE_QUESTIONS` câu đầu từ dev30. Cell in metric và tối đa ba cặp prediction/reference để kiểm tra thủ công.

In [ ]:
dev_questions = json.loads((DATA / 'dev30.questions.json').read_text(encoding='utf-8'))
dev_references = json.loads((DATA / 'dev30.references.json').read_text(encoding='utf-8'))
smoke_ids = list(dev_questions)[:SMOKE_QUESTIONS]
if not smoke_ids:
    raise RuntimeError('Dev smoke không có câu hỏi.')
SMOKE_QUESTIONS_PATH = RUN_ROOT / 'smoke.questions.json'
SMOKE_REFERENCES_PATH = RUN_ROOT / 'smoke.references.json'
SMOKE_RETRIEVAL = RUN_ROOT / 'smoke.retrieval.json'
SMOKE_PREDICTIONS = RUN_ROOT / 'smoke.predictions.json'
SMOKE_METRICS = RUN_ROOT / 'smoke.metrics.json'
SMOKE_QUESTIONS_PATH.write_text(json.dumps({key: dev_questions[key] for key in smoke_ids}, ensure_ascii=False, indent=2), encoding='utf-8')
SMOKE_REFERENCES_PATH.write_text(json.dumps({key: dev_references[key] for key in smoke_ids}, ensure_ascii=False, indent=2), encoding='utf-8')

run('retrieve', '--questions', SMOKE_QUESTIONS_PATH, '--index', INDEX, '--output', SMOKE_RETRIEVAL)
run('generate', '--questions', SMOKE_QUESTIONS_PATH, '--retrieval', SMOKE_RETRIEVAL, '--output', SMOKE_PREDICTIONS, '--mode', GENERATION_MODE)
run('evaluate', '--predictions', SMOKE_PREDICTIONS, '--references', SMOKE_REFERENCES_PATH, '--output', SMOKE_METRICS, '--label', 'pipeline_smoke')

predictions = json.loads(SMOKE_PREDICTIONS.read_text(encoding='utf-8'))
metrics = json.loads(SMOKE_METRICS.read_text(encoding='utf-8'))
print('SMOKE PASSED:', json.dumps(metrics, ensure_ascii=False, indent=2))
for key in smoke_ids[:3]:
    print('\nID:', key)
    print('Question:', dev_questions[key]['question'])
    print('Prediction:', predictions[key]['answer'])
    print('Reference:', dev_references[key])

## Kết quả mong đợi

Smoke đạt khi cell cuối in `SMOKE PASSED`, có đủ prediction cho mọi ID, không có exception và các artifact nằm trong `/kaggle/working/legalqa_smoke_v1`. Metric thấp với corpus mẫu là bình thường.